# Phase 3 · Jev disease gate and domain features

Phase 3 first asks whether each selected species–abstract pair provides human disease evidence. Pairs at or above the configured 0.75 confidence threshold then receive all applicable features for their organism group in one additional Jev request. Successful outputs resume only when the run, abstract hash, model, question version, and feature profile still match.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.decisions.jev import openrouter_key_available
from pathogen_atlas_v3.pipelines.phase3_decisions import phase3_preflight, run_phase3

CONFIG_PATH = "configs/default.yaml"
CONFIG = load_yaml(CONFIG_PATH)
GROUPS = selected_organism_groups(CONFIG)
print("Selected groups:", GROUPS)
print("Model:", CONFIG["decisions"]["model"])
print("Disease threshold:", CONFIG["decisions"]["pathogen_threshold"])
print("Feature yes threshold:", CONFIG["decisions"]["feature_yes_threshold"])
print("OpenRouter key available in environment or Keychain:", openrouter_key_available())
print("Concurrency:", CONFIG["decisions"]["max_workers"], "| progress/checkpoint interval:", CONFIG["logging"]["progress_every"])
print("Decision ledger:", CONFIG["paths"]["decision_ledger"])
print("Raw Jev responses:", CONFIG["paths"]["raw_responses"])

## Preflight · no Jev calls

Review the selected pair count, pending gate and feature calls, estimated tokens, and configured USD estimate before running. The Jev runner prints per-stage progress, failure counts, ETA, and remaining OpenRouter balance.

In [ ]:
PREFLIGHT = phase3_preflight(CONFIG_PATH)
display(pd.DataFrame([PREFLIGHT]))

## Run or resume Jev

`MAX_CALLS` limits newly submitted species–PMID decisions in each Jev stage. Keep the default bounded value for a pilot. Use `None` only when the preflight and available balance support the complete selected-domain run. Failed responses are recorded and can be retried by rerunning.

In [ ]:
RUN_PHASE = False
MAX_CALLS = 500
LOG_PROGRESS = True

if RUN_PHASE:
    if not openrouter_key_available():
        raise RuntimeError("Set OPENROUTER_API_KEY or add the existing OpenRouter Keychain item.")
    DECISIONS = run_phase3(CONFIG_PATH, max_calls=MAX_CALLS, log=LOG_PROGRESS)
    display({"gate_rows": len(DECISIONS["gate"]), "feature_rows": len(DECISIONS["features"]), "selected_groups": GROUPS})
else:
    print("Preflight only. Set RUN_PHASE=True to run or resume Jev decisions.")

## Release policy

The probability thresholds remain provisional. Phase 4 marks candidate evidence as accepted only when `decisions.release_evaluation_approved` is true after labelled evaluation.